# SaaS Funnel Conversion Analysis

**Objetivo:** Analisar as taxas de conversão em cada etapa do funil de trials de um produto B2B SaaS, identificar gargalos e propor hipóteses de melhoria.

**Tecnologias:** Python · Pandas · Matplotlib · Seaborn

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

df = pd.read_csv('funnel_data.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Visao Geral do Dataset

In [ ]:
print("Tipos de dados:")
print(df.dtypes)
print()
print("Valores nulos:")
print(df.isnull().sum())
print()
print("Distribuicao por segmento:")
print(df['segment'].value_counts())

## 2. Funil Geral de Conversao

In [ ]:
stages = ['visited_landing_page', 'started_trial', 'activated',
          'used_key_feature', 'converted_to_paid']

stage_labels = ['Visitou LP', 'Iniciou Trial', 'Ativou', 'Usou Feature-chave', 'Converteu']

totals = df[stages].sum()
rates  = (totals / totals.iloc[0] * 100).round(1)

funnel_df = pd.DataFrame({
    'Etapa': stage_labels,
    'Usuarios': totals.values,
    'Taxa Acumulada (%)': rates.values
})

print(funnel_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots()
colors = sns.color_palette('Blues_d', len(stage_labels))
bars = ax.barh(stage_labels[::-1], totals.values[::-1], color=colors)

for bar, val, rate in zip(bars, totals.values[::-1], rates.values[::-1]):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{int(val)} usuarios ({rate}%)', va='center', fontsize=10)

ax.set_xlabel('Numero de Usuarios')
ax.set_title('Funil de Conversao - Visao Geral')
ax.set_xlim(0, totals.max() * 1.3)
plt.tight_layout()
plt.savefig('funnel_overview.png', dpi=150)
plt.show()

## 3. Drop-off por Etapa

In [ ]:
dropoffs = []
for i in range(1, len(stages)):
    prev = totals.iloc[i-1]
    curr = totals.iloc[i]
    dropoff = ((prev - curr) / prev * 100).round(1)
    dropoffs.append({
        'De -> Para': f'{stage_labels[i-1]} -> {stage_labels[i]}',
        'Perdidos': int(prev - curr),
        'Drop-off (%)': dropoff
    })

dropoff_df = pd.DataFrame(dropoffs)
print(dropoff_df.to_string(index=False))
print()
print(f"Maior gargalo: {dropoff_df.loc[dropoff_df['Drop-off (%)'].idxmax(), 'De -> Para']}")

## 4. Conversao por Segmento

In [ ]:
seg_conv = df.groupby('segment')['converted_to_paid'].agg(['sum','count'])
seg_conv['conversion_rate'] = (seg_conv['sum'] / seg_conv['count'] * 100).round(1)
seg_conv.columns = ['Convertidos', 'Total', 'Taxa de Conversao (%)']
print(seg_conv)

fig, ax = plt.subplots()
sns.barplot(data=seg_conv.reset_index(), x='segment', y='Taxa de Conversao (%)', ax=ax)
ax.set_title('Taxa de Conversao por Segmento')
ax.set_xlabel('Segmento')
plt.tight_layout()
plt.savefig('conversion_by_segment.png', dpi=150)
plt.show()

## 5. Conversao por Canal de Aquisicao

In [ ]:
channel_conv = df.groupby('channel')['converted_to_paid'].agg(['sum','count'])
channel_conv['conversion_rate'] = (channel_conv['sum'] / channel_conv['count'] * 100).round(1)
channel_conv = channel_conv.sort_values('conversion_rate', ascending=False)
channel_conv.columns = ['Convertidos', 'Total', 'Taxa de Conversao (%)']
print(channel_conv)

fig, ax = plt.subplots()
sns.barplot(data=channel_conv.reset_index(), x='channel', y='Taxa de Conversao (%)',
            order=channel_conv.index, ax=ax)
ax.set_title('Taxa de Conversao por Canal')
ax.set_xlabel('Canal')
plt.tight_layout()
plt.savefig('conversion_by_channel.png', dpi=150)
plt.show()

## 6. Conclusoes e Hipoteses de Melhoria

### Principais achados:
- O maior drop-off ocorre em **Visitou LP -> Iniciou Trial**, sugerindo fricao ou falta de relevancia na landing page.
- Usuarios **Enterprise** apresentam maior taxa de conversao final, apesar do menor volume.
- O canal **Referral** tende a trazer usuarios com melhor conversao.

### Hipoteses para A/B teste:
1. **LP:** Testar headline focada em resultado vs. feature para reduzir drop-off no topo do funil.
2. **Onboarding:** Guiar o usuario a usar a key feature nas primeiras 24h aumenta ativacao.
3. **Email nurturing:** Sequencia segmentada por plano durante o trial aumenta conversao final.

---
*Projeto desenvolvido por Gabrielle Magalhaes Soares | github.com/maggabrielle*